In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [4]:
class CricketState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    strike_rate: float
    balls_per_boundary: float
    boundary_percentage: float

    summary: str

In [5]:
def calculate_strike_rate(state: CricketState):
    strike_rate = (state["runs"] / state["balls"]) * 100
    return {"strike_rate": round(strike_rate, 2)}


def calculate_balls_per_boundary(state: CricketState):
    boundaries = state["fours"] + state["sixes"]
    balls_per_boundary = state["balls"] / boundaries
    return {"balls_per_boundary": round(balls_per_boundary, 2)}


def calculate_boundary_percentage(state: CricketState):
    boundary_runs = (state["fours"] * 4) + (state["sixes"] * 6)
    percentage = (boundary_runs / state["runs"]) * 100
    return {"boundary_percentage": round(percentage, 2)}

In [6]:
def generate_summary(state: CricketState):
    summary = f"""
    Cricket Batting Summary
    -----------------------
    Runs: {state['runs']}
    Balls: {state['balls']}
    Fours: {state['fours']}
    Sixes: {state['sixes']}

    Strike Rate: {state['strike_rate']}
    Balls per Boundary: {state['balls_per_boundary']}
    Boundary Run Percentage: {state['boundary_percentage']}%
    """
    return {"summary": summary}

In [7]:
graph = StateGraph(CricketState)

graph.add_node("strike_rate", calculate_strike_rate)
graph.add_node("balls_per_boundary", calculate_balls_per_boundary)
graph.add_node("boundary_percentage", calculate_boundary_percentage)
graph.add_node("summary", generate_summary)

# Parallel branches
graph.add_edge(START, "strike_rate")
graph.add_edge(START, "balls_per_boundary")
graph.add_edge(START, "boundary_percentage")

# Join all branches before summary
graph.add_edge(
    ["strike_rate", "balls_per_boundary", "boundary_percentage"],
    "summary"
)

graph.add_edge("summary", END)

workflow = graph.compile()

In [8]:
initial_state = {
    "runs": 80,
    "balls": 50,
    "fours": 8,
    "sixes": 4
}

final_state = workflow.invoke(initial_state)

print(final_state["summary"])


    Cricket Batting Summary
    -----------------------
    Runs: 80
    Balls: 50
    Fours: 8
    Sixes: 4

    Strike Rate: 160.0
    Balls per Boundary: 4.17
    Boundary Run Percentage: 70.0%
    
